In [1]:
!pip install ultralytics easyocr opencv-python numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 30.3 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

# import zipfile
# import os

# zip_path = "/content/drive/MyDrive/license_plate_detection.zip"   # change this
# extract_path = "/content/license_plate_detection"  # where you want it extracted

# os.makedirs(extract_path, exist_ok=True)

# with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#     zip_ref.extractall(extract_path)

# print("Extraction complete.")

Mounted at /content/drive


In [7]:
from ultralytics import YOLO
import easyocr
import cv2
import re
import torch
import numpy as np

# =========================
# CONFIG
# =========================
MODEL_PATH = "/content/drive/MyDrive/license_plate_detector.pt"
VIDEO_PATH = "/content/drive/MyDrive/final_output_number_plate_test.mp4"
OUTPUT_VIDEO_PATH = "/content/output_annotated.mp4"

# Detection
CONF = 0.40
IOU = 0.45
IMGSZ = 1280

# OCR
OCR_LANGS = ["en"]
OCR_ALLOWLIST = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"

# Plate filtering
MIN_PLATE_W = 50
MIN_PLATE_H = 18
MIN_PLATE_AREA = 1200

# Crop padding
PAD_X = 0.10
PAD_Y = 0.20

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# LOAD MODEL + OCR
# =========================
model = YOLO(MODEL_PATH)
reader = easyocr.Reader(OCR_LANGS, gpu=torch.cuda.is_available())

# =========================
# HELPERS
# =========================
def expand_box(x1, y1, x2, y2, w, h, pad_x=0.10, pad_y=0.20):
    bw = x2 - x1
    bh = y2 - y1

    x1 = int(max(0, x1 - bw * pad_x))
    y1 = int(max(0, y1 - bh * pad_y))
    x2 = int(min(w - 1, x2 + bw * pad_x))
    y2 = int(min(h - 1, y2 + bh * pad_y))
    return x1, y1, x2, y2

def sharpen(img):
    kernel = np.array([[0, -1,  0],
                       [-1, 5, -1],
                       [0, -1,  0]], dtype=np.float32)
    return cv2.filter2D(img, -1, kernel)

def unsharp_mask(img, sigma=1.2, amount=1.8):
    blurred = cv2.GaussianBlur(img, (0, 0), sigma)
    sharp = cv2.addWeighted(img, 1 + amount, blurred, -amount, 0)
    return np.clip(sharp, 0, 255).astype(np.uint8)

def clean_plate_text(text: str) -> str:
    text = text.upper().strip()
    text = re.sub(r"[^A-Z0-9]", "", text)
    return text

def plausible_plate(text: str) -> bool:
    if len(text) < 5 or len(text) > 10:
        return False
    if not re.search(r"[A-Z]", text):
        return False
    if not re.search(r"[0-9]", text):
        return False
    return True

def score_candidate(text, conf):
    text = clean_plate_text(text)
    if not text:
        return -1.0, ""

    score = conf

    if plausible_plate(text):
        score += 1.0

    score += min(len(text), 10) * 0.08

    if len(set(text)) <= 2:
        score -= 0.5

    return score, text

def clean_plate_components(
    img,
    threshold=127,
    invert=False,
    min_area=20,
    max_area=2000,
    min_height=15,
    max_height_ratio=0.95,
    min_width=2,
    max_width_ratio=0.25,
    min_aspect=0.1,
    max_aspect=1.2,
):
    if img is None:
        raise ValueError("Input image is None")

    _, bw = cv2.threshold(img, threshold, 255, cv2.THRESH_BINARY)
    if invert:
        bw = 255 - bw

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(bw, connectivity=8)

    h, w = bw.shape
    clean = np.zeros_like(bw)

    for i in range(1, num_labels):
        ww = stats[i, cv2.CC_STAT_WIDTH]
        hh = stats[i, cv2.CC_STAT_HEIGHT]
        area = stats[i, cv2.CC_STAT_AREA]

        aspect = ww / float(hh) if hh > 0 else 0

        if area < min_area or area > max_area:
            continue
        if hh < min_height or hh > h * max_height_ratio:
            continue
        if ww < min_width or ww > w * max_width_ratio:
            continue
        if aspect < min_aspect or aspect > max_aspect:
            continue

        clean[labels == i] = 255

    return clean

def preprocess_plate_crop(plate_bgr):
    # same preprocessing idea as your code
    gray = cv2.cvtColor(plate_bgr, cv2.COLOR_BGR2GRAY)

    h, w = gray.shape[:2]
    if w < 100:
        scale = 6
    elif w < 160:
        scale = 4
    else:
        scale = 3

    gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_LANCZOS4)
    den = cv2.bilateralFilter(gray, d=9, sigmaColor=40, sigmaSpace=40)

    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    con = clahe.apply(den)

    sharp_img = unsharp_mask(con)

    bw = cv2.adaptiveThreshold(
        sharp_img, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        31, 8
    )

    cleaned = clean_plate_components(bw)

    return {
        "gray_upscaled": gray,
        "sharp": sharp_img,
        "binary": bw,
        "cleaned": cleaned,
    }

def make_ocr_variants(plate_img):
    if plate_img is None:
        raise ValueError("plate_img is None")

    if len(plate_img.shape) == 2:
        gray = plate_img.copy()
    elif len(plate_img.shape) == 3:
        gray = cv2.cvtColor(plate_img, cv2.COLOR_BGR2GRAY)
    else:
        raise ValueError(f"Unsupported image shape: {plate_img.shape}")

    h, w = gray.shape[:2]
    scale = 3 if w < 120 else 2
    gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)

    gray = cv2.bilateralFilter(gray, 9, 75, 75)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    sharp = sharpen(clahe)

    variants = [sharp]

    thr1 = cv2.adaptiveThreshold(
        sharp, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 11
    )
    thr2 = cv2.adaptiveThreshold(
        sharp, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 31, 9
    )
    _, otsu = cv2.threshold(sharp, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    variants.extend([thr1, thr2, otsu, cv2.bitwise_not(otsu)])
    return variants

def run_ocr_on_crop(plate_img):
    variants = make_ocr_variants(plate_img)

    best_score = -1.0
    best_text = ""

    for var in variants:
        results = reader.readtext(
            var,
            detail=1,
            paragraph=False,
            decoder="greedy",
            allowlist=OCR_ALLOWLIST
        )

        for item in results:
            text = item[1]
            conf = float(item[2])
            score, cleaned = score_candidate(text, conf)
            if score > best_score:
                best_score = score
                best_text = cleaned

    return best_text, best_score

def get_best_plate_text(plate_crop):
    processed = preprocess_plate_crop(plate_crop)

    best_text = ""
    best_score = -1.0
    best_variant_name = None

    for name, img in processed.items():
        text, score = run_ocr_on_crop(img)
        if score > best_score:
            best_score = score
            best_text = text
            best_variant_name = name

    return best_text, best_score, best_variant_name

def draw_label(img, text, x1, y1, color=(0, 255, 0)):
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.8
    thickness = 2

    (tw, th), baseline = cv2.getTextSize(text, font, font_scale, thickness)

    box_y1 = max(0, y1 - th - baseline - 10)
    box_y2 = y1
    box_x1 = x1
    box_x2 = x1 + tw + 10

    cv2.rectangle(img, (box_x1, box_y1), (box_x2, box_y2), color, -1)
    cv2.putText(
        img,
        text,
        (box_x1 + 5, box_y2 - 5),
        font,
        font_scale,
        (0, 0, 0),
        thickness,
        cv2.LINE_AA
    )

# =========================
# VIDEO INFERENCE + WRITE OUTPUT
# =========================
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise FileNotFoundError(f"Could not open video: {VIDEO_PATH}")

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(
    OUTPUT_VIDEO_PATH,
    fourcc,
    fps if fps > 0 else 25,
    (frame_width, frame_height)
)

frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_idx += 1
    h, w = frame.shape[:2]
    annotated = frame.copy()

    results = model.predict(
        source=frame,
        conf=CONF,
        iou=IOU,
        imgsz=IMGSZ,
        save=False,
        verbose=False
    )

    result = results[0]

    if result.boxes is not None and len(result.boxes) > 0:
        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            det_conf = float(box.conf[0])

            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            x1, y1, x2, y2 = expand_box(x1, y1, x2, y2, w, h, PAD_X, PAD_Y)

            bw = x2 - x1
            bh = y2 - y1
            area = bw * bh

            if bw < MIN_PLATE_W or bh < MIN_PLATE_H or area < MIN_PLATE_AREA:
                continue

            plate_crop = frame[y1:y2, x1:x2]
            if plate_crop.size == 0:
                continue

            crop_h, crop_w = plate_crop.shape[:2]
            if crop_w < 80 or crop_h < 25:
                continue

            best_text, best_score, best_variant = get_best_plate_text(plate_crop)

            # draw plate box
            cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)


            # build label
            if best_text:
                # label = f"{best_text} | det:{det_conf:.2f} | ocr:{best_score:.2f}"
                label = f"{best_text}"
            else:
                label = f"PLATE | det:{det_conf:.2f}"

            draw_label(annotated, label, x1, y1, color=(0, 255, 0))

    writer.write(annotated)

cap.release()
writer.release()
cv2.destroyAllWindows()

print(f"Saved output video to: {OUTPUT_VIDEO_PATH}")

Saved output video to: /content/output_annotated.mp4


In [8]:
from google.colab.files import download

download("output_annotated.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# main code is below

In [ ]:
# main code is below

In [ ]:
from ultralytics import YOLO
import easyocr
import cv2
import re
import torch
import numpy as np
from collections import deque
from google.colab.patches import cv2_imshow

# =========================
# CONFIG
# =========================
MODEL_PATH = "/content/drive/MyDrive/license_plate_detector.pt"
VIDEO_PATH = "/content/drive/MyDrive/final_output_number_plate_test.mp4"
OUTPUT_VIDEO_PATH = "/content/output_annotated.mp4"

# Detection
CONF = 0.40
IOU = 0.45
IMGSZ = 1280

# OCR
OCR_LANGS = ["en"]
OCR_ALLOWLIST = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"

# Plate filtering
MIN_PLATE_W = 50
MIN_PLATE_H = 18
MIN_PLATE_AREA = 1200

# Crop padding
PAD_X = 0.10
PAD_Y = 0.20

# Temporal smoothing
# TRACK_DISTANCE_THRESH = 80
# TRACK_MAX_AGE = 10
# TEXT_VOTES_TO_LOCK = 2


TEXT_VOTES_TO_LOCK = 4
TRACK_MAX_AGE = 15
TRACK_DISTANCE_THRESH = 60


# Colab display
SHOW_EVERY_N_FRAMES = 15

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# LOAD MODEL + OCR
# =========================
model = YOLO(MODEL_PATH)
reader = easyocr.Reader(OCR_LANGS, gpu=torch.cuda.is_available())

# =========================
# HELPERS
# =========================
def clean_plate_text(text: str) -> str:
    text = text.upper().strip()
    text = re.sub(r"[^A-Z0-9]", "", text)
    return text

def plausible_plate(text: str) -> bool:
    if len(text) < 5 or len(text) > 10:
        return False
    if not re.search(r"[A-Z]", text):
        return False
    if not re.search(r"[0-9]", text):
        return False
    return True

def expand_box(x1, y1, x2, y2, w, h, pad_x=0.10, pad_y=0.20):
    bw = x2 - x1
    bh = y2 - y1

    x1 = int(max(0, x1 - bw * pad_x))
    y1 = int(max(0, y1 - bh * pad_y))
    x2 = int(min(w - 1, x2 + bw * pad_x))
    y2 = int(min(h - 1, y2 + bh * pad_y))
    return x1, y1, x2, y2

def sharpen(img):
    kernel = np.array([[0, -1,  0],
                       [-1, 5, -1],
                       [0, -1,  0]], dtype=np.float32)
    return cv2.filter2D(img, -1, kernel)

def make_ocr_variants(plate_bgr):
    gray = cv2.cvtColor(plate_bgr, cv2.COLOR_BGR2GRAY)

    h, w = gray.shape[:2]
    scale = 3 if w < 120 else 2
    gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)

    gray = cv2.bilateralFilter(gray, 9, 75, 75)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    sharp = sharpen(clahe)

    variants = []

    variants.append(sharp)

    thr1 = cv2.adaptiveThreshold(
        sharp, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 11
    )
    variants.append(thr1)

    thr2 = cv2.adaptiveThreshold(
        sharp, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 31, 9
    )
    variants.append(thr2)

    _, otsu = cv2.threshold(sharp, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    variants.append(otsu)

    variants.append(cv2.bitwise_not(otsu))

    return variants

def score_candidate(text, conf):
    text = clean_plate_text(text)
    if not text:
        return -1.0, ""

    score = conf

    if plausible_plate(text):
        score += 1.0

    score += min(len(text), 10) * 0.08

    # penalize suspicious repeats like IIIII / 00000 / very weird strings
    if len(set(text)) <= 2:
        score -= 0.5

    return score, text

def run_ocr_on_crop(plate_bgr):
    variants = make_ocr_variants(plate_bgr)

    best_score = -1.0
    best_text = ""

    for var in variants:
        results = reader.readtext(
            var,
            detail=1,
            paragraph=False,
            decoder="greedy",
            allowlist=OCR_ALLOWLIST
        )

        for item in results:
            text = item[1]
            conf = float(item[2])
            score, cleaned = score_candidate(text, conf)
            if score > best_score:
                best_score = score
                best_text = cleaned

    return best_text

def draw_plate_result(image_bgr, x1, y1, x2, y2, conf, plate_text):
    cv2.rectangle(image_bgr, (x1, y1), (x2, y2), (0, 255, 0), 2)

    label = f"{conf:.2f}"
    if plate_text:
        label += f" | {plate_text}"

    cv2.putText(
        image_bgr,
        label,
        (x1, max(25, y1 - 10)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2,
        cv2.LINE_AA
    )
    return image_bgr

# =========================
# SIMPLE TEMPORAL STABILIZER
# =========================
tracks = []
next_track_id = 0

def center_of_box(box):
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2, (y1 + y2) / 2)

def dist(c1, c2):
    return ((c1[0] - c2[0]) ** 2 + (c1[1] - c2[1]) ** 2) ** 0.5

def update_tracks(detections):
    global tracks, next_track_id

    for t in tracks:
        t["age"] += 1

    for det in detections:
        box = det["box"]
        text = det["text"]
        conf = det["conf"]
        c = center_of_box(box)

        best_track = None
        best_d = 1e9

        for t in tracks:
            d = dist(c, t["center"])
            if d < TRACK_DISTANCE_THRESH and d < best_d:
                best_d = d
                best_track = t

        if best_track is None:
            new_track = {
                "id": next_track_id,
                "center": c,
                "box": box,
                "age": 0,
                "history": deque(maxlen=12),
                "locked_text": ""
            }
            if text:
                new_track["history"].append(text)
            tracks.append(new_track)
            det["stable_text"] = text
            next_track_id += 1
        else:
            best_track["center"] = c
            best_track["box"] = box
            best_track["age"] = 0
            if text:
                best_track["history"].append(text)

            if best_track["history"]:
                counts = {}
                for txt in best_track["history"]:
                    counts[txt] = counts.get(txt, 0) + 1
                best_txt = max(counts, key=counts.get)
                if counts[best_txt] >= TEXT_VOTES_TO_LOCK:
                    best_track["locked_text"] = best_txt

            det["stable_text"] = best_track["locked_text"] if best_track["locked_text"] else text

    tracks = [t for t in tracks if t["age"] <= TRACK_MAX_AGE]

# =========================
# VIDEO INFERENCE
# =========================
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise FileNotFoundError(f"Could not open video: {VIDEO_PATH}")

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(
    OUTPUT_VIDEO_PATH,
    fourcc,
    fps if fps > 0 else 25,
    (frame_width, frame_height)
)

frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_idx += 1
    h, w = frame.shape[:2]
    annotated = frame.copy()

    results = model.predict(
        source=frame,
        conf=CONF,
        iou=IOU,
        imgsz=IMGSZ,
        save=False,
        verbose=False
    )

    result = results[0]
    detections = []

    if result.boxes is not None and len(result.boxes) > 0:
        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            det_conf = float(box.conf[0])

            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            x1, y1, x2, y2 = expand_box(x1, y1, x2, y2, w, h, PAD_X, PAD_Y)

            bw = x2 - x1
            bh = y2 - y1
            area = bw * bh

            if bw < MIN_PLATE_W or bh < MIN_PLATE_H or area < MIN_PLATE_AREA:
                continue

            plate_crop = frame[y1:y2, x1:x2]
            if plate_crop.size == 0:
                continue

            # cv2_imshow(plate_crop)

            plate_text = run_ocr_on_crop(plate_crop)

            detections.append({
                "box": (x1, y1, x2, y2),
                "conf": det_conf,
                "text": plate_text
            })

    update_tracks(detections)

    for det in detections:
        x1, y1, x2, y2 = det["box"]
        det_conf = det["conf"]
        stable_text = det.get("stable_text", det["text"])

        annotated = draw_plate_result(
            annotated, x1, y1, x2, y2, det_conf, stable_text
        )

    cv2.putText(
        annotated,
        f"Frame: {frame_idx}",
        (20, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.0,
        (0, 255, 255),
        2,
        cv2.LINE_AA
    )

    writer.write(annotated)

    # if frame_idx % SHOW_EVERY_N_FRAMES == 0:
    #     cv2_imshow(annotated)

cap.release()
writer.release()
cv2.destroyAllWindows()

print(f"Saved output video to: {OUTPUT_VIDEO_PATH}")

Saved output video to: /content/output_annotated.mp4
